# LifeMate Agent — Part C: Agent LLM & Sequential Tool-Calling (Colab / Kaggle)
**Team 3 — Member 3 Specialist Guide**

This notebook runs the LLM Agent pipeline with:
- **Qwen2.5-3B-Instruct** (or 7B with 4-bit AWQ / bfloat16 on T4 / T4x2)
- **Tool Filtering per Sample** (reducing 147 tools down to installed apps)
- **Pre-Call Validation** (catches schema errors before sending to ToolEnv)
- **Multi-turn Read → Write Execution Loop**
- **Offline Evaluation on Dev Set**

## 1. Setup Environment (Colab / Kaggle T4)

In [ ]:
!nvidia-smi
# Install dependencies (vllm is ideal for fast inference, or transformers + accelerate)
!pip install -q transformers accelerate sentencepiece vllm

## 2. Load Tools Catalog & Filter Logic

In [ ]:
import json, re, unicodedata, time
from pathlib import Path

CATALOG_PATH = Path("./train+public_test/train/train_tools_catalog.json")
with open(CATALOG_PATH, encoding="utf-8") as f:
    catalog = json.load(f)

print(f"Loaded catalog: {len(catalog.get('apps', []))} apps")

# Map tool name -> spec
tool_specs = {}
for app in catalog.get("apps", []):
    for tool in app.get("tools", []):
        tool_specs[tool["name"]] = tool
print(f"Total tools indexed: {len(tool_specs)}")

## 3. Tool Filtering per Sample

In [ ]:
def filter_tools_for_sample(sample: dict, catalog: dict) -> list[dict]:
    installed = set(sample.get("context", {}).get("installed_apps", {}).keys())
    filtered = []
    for app in catalog.get("apps", []):
        if app["name"] in installed or app["name"] == "sys":
            filtered.append(app)
    return filtered

# Test filter on a dummy context
dummy_sample = {"context": {"installed_apps": {"smart_home": {}, "contacts": {}}}}
f_apps = filter_tools_for_sample(dummy_sample, catalog)
print("Filtered apps:", [a["name"] for a in f_apps])

## 4. Load Qwen2.5-3B-Instruct Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Model loaded into VRAM!")

## 5. System Prompt & Tool Calling Format

In [ ]:
SYSTEM_PROMPT = """Bạn là trợ lý LifeMate thông minh. Nhiệm vụ của bạn là gọi các công cụ (tools) phù hợp để thực thi yêu cầu người dùng.
Quy tắc quan trọng:
1. Chỉ gọi các công cụ thuộc các app đã cài đặt (installed_apps).
2. Khi cần ID để thực hiện lệnh (device_id, contact_id, reminder_id...), BẮT BUỘC phải gọi lệnh đọc (search, list, get) trước để lấy ID chính xác.
3. Nếu thiếu thông tin bắt buộc mà không tìm thấy trong hệ thống, hãy gọi `sys.ask_missing_info`.
4. Nếu yêu cầu không thuộc bất kỳ app nào được cài, KHÔNG gọi bất kỳ tool nào (OOS).
5. Xuất định dạng JSON dạng: ```json
[
  {"name": "tool_name", "arguments": {...}}
]
```
"""
print("System prompt ready:", len(SYSTEM_PROMPT), "characters")

## 6. End-to-End Test on Dev Sample

In [ ]:
user_query = "bật đèn phòng khách giúp tôi"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Yêu cầu: {user_query}\nThời gian hiện tại: 2026-11-02T19:00:00+07:00\nApp đã cài: smart_home"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Greedy decoding for deterministic tool calls
    )

response = tokenizer.decode(generated_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("Agent Response:")
print(response)